In [0]:
%restart_python
!pip install tldextract


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/107.4 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
import tldextract

def extract_domain_parts(url):
    """Extract domain parts using tldextract"""
    try:
        ext = tldextract.extract(url)
        registered_domain = '.'.join([part for part in [ext.domain, ext.suffix] if part])
        return registered_domain
    except Exception as e:
        return None

# Register UDF
extract_domain_udf = udf(extract_domain_parts, StringType())

In [0]:
# Read Common Crawl and get unique domains
df_common_crawl = spark \
    .read \
    .option("basePath", "s3://commoncrawl/cc-index/table/cc-main/warc/") \
    .parquet(f"s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2025-18/subset=warc/") \
    .filter(F.col("fetch_status") == 200)

common_crawl_domains = df_common_crawl \
    .select("url_host_registered_domain") \
    .filter(F.col("url_host_registered_domain").isNotNull()) \
    .distinct()

# Cache it since we'll use it multiple times
common_crawl_domains.cache()
print(f"Common crawl domains loaded and cached")

Common crawl domains loaded and cached


In [0]:
# Read Site Visit data
df_site_visit = spark \
    .read \
    .option("basePath", "s3://mntn-data-archive-prod/signals/site_visit_signal/") \
    .parquet(f"s3://mntn-data-archive-prod/signals/site_visit_signal/dt=2025-06-01/hh=00/") \
    .filter(F.col("data_source_id") != 23)

print(f"Site visit raw records: {df_site_visit.count():,}")

Site visit raw records: 56,986,221


In [0]:
# Extract unique domains from site visits
site_visit_domains = df_site_visit \
    .withColumn("domain", extract_domain_udf("url")) \
    .select("domain") \
    .filter(F.col("domain").isNotNull()) \
    .distinct()

# Cache it
site_visit_domains.cache()
print("Site visit domains extracted and cached")

# Show sample to verify extraction worked
print("\nSample extracted domains:")
site_visit_domains.show(10, truncate=False)

Site visit domains extracted and cached

Sample extracted domains:
+--------------------+
|domain              |
+--------------------+
|explorewithalec.com |
|poshpeanut.com      |
|coterie.com         |
|officeboffins.co.uk |
|trysnow.com         |
|garagegymreviews.com|
|mrishtanna.com      |
|wafflegame.net      |
|officer.com         |
|dekohochzeit.info   |
+--------------------+
only showing top 10 rows


In [0]:
# Get counts
print("Counting unique domains...")
site_domains_count = site_visit_domains.count()
crawl_domains_count = common_crawl_domains.count()

print(f"\nSite visit unique domains: {site_domains_count:,}")
print(f"Common crawl unique domains: {crawl_domains_count:,}")

Counting unique domains...

Site visit unique domains: 155,746
Common crawl unique domains: 38,869,252


In [0]:
# With 155K site domains vs 38.8M crawl domains, we should:
# 1. Collect the smaller site_visit_domains to driver
# 2. Broadcast it when joining

print("Collecting site visit domains to driver (155K domains)...")
site_domains_list = site_visit_domains.collect()
site_domains_set = set([row.domain for row in site_domains_list])
print(f"Collected {len(site_domains_set):,} unique site visit domains")

# Now broadcast the site domains for efficient join
from pyspark.sql.functions import broadcast

print("\nCalculating coverage using broadcast join...")
domains_in_common_crawl = broadcast(site_visit_domains).join(
    common_crawl_domains,
    site_visit_domains.domain == common_crawl_domains.url_host_registered_domain,
    "inner"
).select("domain").distinct()

# Cache the result for reuse
domains_in_common_crawl.cache()

# Count domains
domains_in_both_count = domains_in_common_crawl.count()
domains_only_in_site_count = len(site_domains_set) - domains_in_both_count

print(f"\n=== COVERAGE RESULTS ===")
print(f"Total unique domains in site visits: {len(site_domains_set):,}")
print(f"Domains found in Common Crawl: {domains_in_both_count:,}")
print(f"Domains NOT in Common Crawl: {domains_only_in_site_count:,}")
print(f"\nCoverage: {(domains_in_both_count / len(site_domains_set) * 100):.2f}% of our domains are in Common Crawl")
print(f"Not covered: {(domains_only_in_site_count / len(site_domains_set) * 100):.2f}% of our domains are NOT in Common Crawl")

Collected 155,746 unique site visit domains

Calculating coverage using broadcast join...

=== COVERAGE RESULTS ===
Total unique domains in site visits: 155,746
Domains found in Common Crawl: 105,916
Domains NOT in Common Crawl: 49,830

Coverage: 68.01% of our domains are in Common Crawl
Not covered: 31.99% of our domains are NOT in Common Crawl


In [0]:
# Find domains that are NOT in Common Crawl
print("Finding domains not covered by Common Crawl...")

domains_not_in_crawl = broadcast(site_visit_domains).join(
    common_crawl_domains,
    site_visit_domains.domain == common_crawl_domains.url_host_registered_domain,
    "left_anti"  # This gives us domains that DON'T match
)

# Cache for reuse
domains_not_in_crawl.cache()

print(f"Domains not in Common Crawl: {domains_not_in_crawl.count():,}")
print("\nSample of domains NOT covered by Common Crawl:")
domains_not_in_crawl.show(30, truncate=False)

Finding domains not covered by Common Crawl...
Domains not in Common Crawl: 49,830

Sample of domains NOT covered by Common Crawl:
+-------------------------+
|domain                   |
+-------------------------+
|0xmedia.co               |
|20.17.0.6                |
|3.14.93.248              |
|3x3custom.com            |
|8book.com.br             |
|aboutsocialanxiety.com   |
|aginmusic.de             |
|aguadesal.com.br         |
|agxco.com                |
|allmodernmommy.com       |
|allprorestorationhelp.com|
|alphaseed.eu             |
|altfootball.com          |
|amhtraducciones.com      |
|andrewduff.net           |
|anurveda.store           |
|aol.co.uk                |
|apieceofanime.com        |
|artofmemory.com          |
|atv.com                  |
|automotiveamerican.com   |
|autonationfordtustin.com |
|avoyellestoday.com       |
|awardsplatform.com       |
|baddiessouth.info        |
|baptist.pl               |
|bcnews168.store          |
|beautynaturalskins.com   |
|

In [0]:
# If you want to see which HIGH-TRAFFIC domains are not covered
# This is more expensive but shows important missing domains

print("Finding high-traffic domains not in Common Crawl...")

# First, get top domains by traffic from site visits
top_site_domains = df_site_visit \
    .withColumn("domain", extract_domain_udf("url")) \
    .groupBy("domain") \
    .agg(F.count("*").alias("visit_count")) \
    .orderBy(F.desc("visit_count")) \
    .limit(10000)  # Top 10k domains by traffic

# Now see which of these are NOT in Common Crawl
top_uncovered = top_site_domains.join(
    broadcast(domains_not_in_crawl),
    "domain",
    "inner"
)

print("\nTop uncovered domains by visit count:")
top_uncovered.show(50, truncate=False)

# Get summary
total_top_uncovered = top_uncovered.agg(F.sum("visit_count")).collect()[0][0]
print(f"\nTotal visits to top uncovered domains: {total_top_uncovered:,}")

Finding high-traffic domains not in Common Crawl...

Top uncovered domains by visit count:
+------------------------+-----------+
|domain                  |visit_count|
+------------------------+-----------+
|aol.com                 |915753     |
|grizly.com              |678059     |
|localhost               |424277     |
|parade.com              |355428     |
|wowhead.com             |335472     |
|thestreet.com           |335398     |
|screenrant.com          |306935     |
|pages.dev               |225318     |
|percentagecalculator.net|186999     |
|athlonsports.com        |162162     |
|gkduniya.com            |151233     |
|fextralife.com          |144902     |
|fabpop.net              |135968     |
|twsn.net                |133387     |
|gamerant.com            |133097     |
|newsweek.com            |131932     |
|entertainmentglow.com   |121327     |
|worldlifestyle.com      |104329     |
|cbr.com                 |91115      |
|conservativebrief.com   |83480      |
|collider.co

In [0]:
print("=== DATASET SCHEMAS ===\n")

print("1. ORIGINAL COMMON CRAWL DATAFRAME (df_common_crawl)")
print(f"   Columns: {df_common_crawl.columns}")
print(f"   Column count: {len(df_common_crawl.columns)}\n")

print("2. COMMON CRAWL DOMAINS (common_crawl_domains)")
print(f"   Columns: {common_crawl_domains.columns}")
print(f"   Schema:")
common_crawl_domains.printSchema()

print("\n3. ORIGINAL SITE VISIT DATAFRAME (df_site_visit)")
print(f"   Columns: {df_site_visit.columns}")
print(f"   Column count: {len(df_site_visit.columns)}\n")

print("4. SITE VISIT DOMAINS (site_visit_domains)")
print(f"   Columns: {site_visit_domains.columns}")
print(f"   Schema:")
site_visit_domains.printSchema()

print("\n5. DOMAINS IN COMMON CRAWL (domains_in_common_crawl)")
print(f"   Columns: {domains_in_common_crawl.columns}")
print(f"   Schema:")
domains_in_common_crawl.printSchema()

print("\n6. DOMAINS NOT IN CRAWL (domains_not_in_crawl)")
print(f"   Columns: {domains_not_in_crawl.columns}")
print(f"   Schema:")
domains_not_in_crawl.printSchema()

=== DATASET SCHEMAS ===

1. ORIGINAL COMMON CRAWL DATAFRAME (df_common_crawl)
   Columns: ['url_surtkey', 'url', 'url_host_name', 'url_host_tld', 'url_host_2nd_last_part', 'url_host_3rd_last_part', 'url_host_4th_last_part', 'url_host_5th_last_part', 'url_host_registry_suffix', 'url_host_registered_domain', 'url_host_private_suffix', 'url_host_private_domain', 'url_host_name_reversed', 'url_protocol', 'url_port', 'url_path', 'url_query', 'fetch_time', 'fetch_status', 'fetch_redirect', 'content_digest', 'content_mime_type', 'content_mime_detected', 'content_charset', 'content_languages', 'content_truncated', 'warc_filename', 'warc_record_offset', 'warc_record_length', 'warc_segment', 'crawl', 'subset']
   Column count: 32

2. COMMON CRAWL DOMAINS (common_crawl_domains)
   Columns: ['url_host_registered_domain']
   Schema:
root
 |-- url_host_registered_domain: string (nullable = true)


3. ORIGINAL SITE VISIT DATAFRAME (df_site_visit)
   Columns: ['uid', 'advertiser_id', 'ip', 'url', 'que

### old common crawl

In [0]:
# Read Common Crawl and get unique domains
df_common_crawl_old = spark \
    .read \
    .option("basePath", "s3://commoncrawl/cc-index/table/cc-main/warc/") \
    .parquet(f"s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2025-05/subset=warc/") \
    .filter(F.col("fetch_status") == 200)

common_crawl_domains_old = df_common_crawl_old \
    .select("url_host_registered_domain") \
    .filter(F.col("url_host_registered_domain").isNotNull()) \
    .distinct()

# Cache it since we'll use it multiple times
common_crawl_domains_old.cache()
print(f"Older common crawl domains loaded and cached")

Older common crawl domains loaded and cached
